# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
from pathlib import Path

# Use the course folder locally, or the attached CSV on Kaggle.
data_path = Path('../data/co2_emissions.csv')
if not data_path.exists():
    matches = list(Path('/kaggle/input').rglob('co2_emissions.csv'))
    if len(matches) != 1:
        raise FileNotFoundError('Attach the course co2_emissions.csv dataset before running.')
    data_path = matches[0]
df = pd.read_csv(data_path)
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

# Sort observations chronologically within each country.
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['Australia' 'Brazil' 'Canada' 'China' 'France' 'Germany' 'India'
 'Indonesia' 'Japan' 'Mexico' 'Russia' 'South Africa' 'South Korea'
 'United Kingdom' 'United States']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [3]:
# Task 1: all 15 countries, with Germany highlighted.
chart_data = df
highlight = 'Germany'
metric = 'CO2_per_capita'
accent = '#176B9B'
axis_label = 'CO₂ per capita (provided values)'
highlight_data = chart_data.loc[chart_data['Country'] == highlight].sort_values('Year')
first, last = highlight_data.iloc[0], highlight_data.iloc[-1]
decline = (1 - last[metric] / first[metric]) * 100
insight_title = f"Germany's CO₂ per capita fell {decline:.0f}% from {int(first['Year'])} to {int(last['Year'])}"
end_label = f"Germany  {last[metric]:.2f}"

# Draw the highlight last, above all context lines.
country_order = [c for c in chart_data['Country'].unique() if c != highlight] + [highlight]
color_map = {c: accent if c == highlight else '#EEEEEE' for c in country_order}
fig = px.line(
    chart_data, x='Year', y=metric, color='Country',
    category_orders={'Country': country_order}, color_discrete_map=color_map,
    labels={'Year': '', metric: axis_label},
    title=insight_title, template='plotly_white', height=480,
)
fig.update_traces(mode='lines', line=dict(width=1), showlegend=False)
fig.update_traces(line=dict(width=3), selector=dict(name=highlight))
fig.add_annotation(
    x=int(last['Year']), y=last[metric], text=end_label,
    showarrow=False, xanchor='left', xshift=10,
    font=dict(color=accent, size=13),
)
fig.update_layout(
    showlegend=False, font=dict(family='Arial', size=13, color='#444444'),
    title=dict(x=0.02, font=dict(size=20)),
    margin=dict(l=65, r=175, t=75, b=55),
    paper_bgcolor='white', plot_bgcolor='white',
)
fig.update_xaxes(
    showgrid=False, showline=False, zeroline=False, ticks='', tickangle=0,
    tickvals=[2000, 2005, 2010, 2015, 2020, 2022],
    range=[int(chart_data['Year'].min()), int(chart_data['Year'].max()) + 0.5],
)
fig.update_yaxes(showgrid=False, showline=False, zeroline=False, rangemode='tozero')
fig1 = fig
fig1.show()


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [4]:
# Task 2: total emissions for every country labelled Asia in the dataset.
chart_data = df.loc[df['Region'] == 'Asia']
highlight = 'China'
metric = 'CO2_Mt'
accent = 'crimson'
axis_label = 'CO₂ emissions (million tonnes)'
highlight_data = chart_data.loc[chart_data['Country'] == highlight].sort_values('Year')
first, last = highlight_data.iloc[0], highlight_data.iloc[-1]
multiple = last[metric] / first[metric]
insight_title = f"China's CO₂ emissions reached {multiple:.1f} times their {int(first['Year'])} level in {int(last['Year'])}"
end_label = f"China  {last[metric]:,.0f} Mt"

# Draw the highlight last, above all context lines.
country_order = [c for c in chart_data['Country'].unique() if c != highlight] + [highlight]
color_map = {c: accent if c == highlight else '#EEEEEE' for c in country_order}
fig = px.line(
    chart_data, x='Year', y=metric, color='Country',
    category_orders={'Country': country_order}, color_discrete_map=color_map,
    labels={'Year': '', metric: axis_label},
    title=insight_title, template='plotly_white', height=480,
)
fig.update_traces(mode='lines', line=dict(width=1), showlegend=False)
fig.update_traces(line=dict(width=3), selector=dict(name=highlight))
fig.add_annotation(
    x=int(last['Year']), y=last[metric], text=end_label,
    showarrow=False, xanchor='left', xshift=10,
    font=dict(color=accent, size=13),
)
fig.update_layout(
    showlegend=False, font=dict(family='Arial', size=13, color='#444444'),
    title=dict(x=0.02, font=dict(size=20)),
    margin=dict(l=65, r=175, t=75, b=55),
    paper_bgcolor='white', plot_bgcolor='white',
)
fig.update_xaxes(
    showgrid=False, showline=False, zeroline=False, ticks='', tickangle=0,
    tickvals=[2000, 2005, 2010, 2015, 2020, 2022],
    range=[int(chart_data['Year'].min()), int(chart_data['Year'].max()) + 0.5],
)
fig.update_yaxes(showgrid=False, showline=False, zeroline=False, rangemode='tozero')
fig.update_yaxes(tickformat=',.0f')
fig2 = fig
fig2.show()


### Interpretation

- **Task 1:** In the supplied dataset, Germany's CO₂ per-capita value falls from **1.25 in 2000 to 0.48 in 2022**, a **61.6% decrease**. All 15 countries remain visible for context.
- **Task 2:** China's total CO₂ emissions rise from **3,513.8 Mt to 12,409.5 Mt**, approximately **3.5 times** the 2000 level. All five countries labelled `Asia` are included: China, India, Indonesia, Japan and South Korea.

Both charts use one coloured, thicker line, thin `#EEEEEE` context lines, a direct endpoint label, and no legend, borders or gridlines.

**Data:** The supplied `co2_emissions.csv` matches the course archive. Values are used as supplied. The CSV does not specify the unit for `CO2_per_capita`, so Task 1 retains the provided scale without assuming a unit or recalculating it. The Our World in Data attribution is from the course notebook.
